<a href="https://colab.research.google.com/github/Junaaid26/nnunet-brain-tumor-segmentation/blob/main/01_nnUNet_BrainTumor_Reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

data_path = "/content/drive/MyDrive/Medical_AI_NnUNet"

print(os.listdir(data_path))

['Task01_BrainTumour.tar']


In [ ]:
import tarfile
import os

tar_path = "/content/drive/MyDrive/Medical_AI_NnUNet/Task01_BrainTumour.tar"
extract_path = "/content/Task01_BrainTumour"

with tarfile.open(tar_path, "r") as tar:
    tar.extractall(extract_path)

print("Extraction complete!")
print(os.listdir(extract_path))

/tmp/ipykernel_1584/4038477034.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_path)


Extraction complete!
['Task01_BrainTumour']


In [ ]:
import os

path = "/content/Task01_BrainTumour/Task01_BrainTumour"

print(os.listdir(path))

['._dataset.json', 'imagesTr', '._labelsTr', '._imagesTr', 'labelsTr', '._imagesTs', 'dataset.json', 'imagesTs']


In [ ]:
import os

base = "/content/Task01_BrainTumour/Task01_BrainTumour"

for folder in ["imagesTr", "imagesTs", "labelsTr"]:
    path = os.path.join(base, folder)
    files = [f for f in os.listdir(path) if not f.startswith("._")]
    print(f"{folder}: {len(files)} files")
    print("First 3:", files[:3])

imagesTr: 484 files
First 3: ['BRATS_310.nii.gz', 'BRATS_347.nii.gz', 'BRATS_041.nii.gz']
imagesTs: 266 files
First 3: ['BRATS_512.nii.gz', 'BRATS_623.nii.gz', 'BRATS_696.nii.gz']
labelsTr: 484 files
First 3: ['BRATS_310.nii.gz', 'BRATS_347.nii.gz', 'BRATS_041.nii.gz']


In [ ]:
!pip install nnunetv2 -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.1 MB/s eta 0:00:00


In [ ]:
!nnUNetv2_plan_and_preprocess --help

usage: nnUNetv2_plan_and_preprocess [-h] [-d D [D ...]] [-fpe FPE]
                                    [-npfp NPFP] [--verify_dataset_integrity]
                                    [--no_pp] [--clean] [-pl PL]
                                    [-gpu_memory_target GPU_MEMORY_TARGET]
                                    [-preprocessor_name PREPROCESSOR_NAME]
                                    [-overwrite_target_spacing OVERWRITE_TARGET_SPACING [OVERWRITE_TARGET_SPACING ...]]
                                    [-overwrite_plans_name OVERWRITE_PLANS_NAME]
                                    [-c C [C ...]] [-np NP [NP ...]]
                                    [--verbose] [--no_pbar]

options:
  -h, --help            show this help message and exit
  -d D [D ...]          [REQUIRED] List of dataset IDs. Example: 2 4 5. This
                        will run fingerprint extraction, experiment planning
                        and preprocessing for these datasets. Can of course
              

In [ ]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

os.makedirs("/content/nnUNet_raw", exist_ok=True)
os.makedirs("/content/nnUNet_preprocessed", exist_ok=True)
os.makedirs("/content/nnUNet_results", exist_ok=True)

print("nnU-Net folders configured:")
print("Raw:", os.environ["nnUNet_raw"])
print("Preprocessed:", os.environ["nnUNet_preprocessed"])
print("Results:", os.environ["nnUNet_results"])

nnU-Net folders configured:
Raw: /content/nnUNet_raw
Preprocessed: /content/nnUNet_preprocessed
Results: /content/nnUNet_results


In [ ]:
!nnUNetv2_convert_MSD_dataset \
-i /content/Task01_BrainTumour/Task01_BrainTumour

In [ ]:
import os

path = "/content/nnUNet_raw/Dataset001_BrainTumour"

print(os.listdir(path))

['imagesTr', 'labelsTr', 'dataset.json', 'imagesTs']


In [ ]:
!nnUNetv2_plan_and_preprocess \
-d 1 \
--verify_dataset_integrity

Fingerprint extraction...
Dataset001_BrainTumour
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 484/484 [08:30<00:00,  1.05s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [138. 169. 138.], 3d_lowres: [138, 169, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'ba

In [ ]:
import os

plans_path = "/content/nnUNet_preprocessed/Dataset001_BrainTumour/nnUNetPlans.json"

print("Plans file exists:", os.path.exists(plans_path))
print("Size:", os.path.getsize(plans_path), "bytes")

Plans file exists: True
Size: 11561 bytes


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
!nnUNetv2_train 1 3d_fullres 0



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-09-16 20:28:57.027683: Using torch.compile...
2026-09-16 20:28:59.839775: do_dummy_2d_data_aug: False
2026-09-16 20:28:59.841082: Creating new 5-fold cross-validation split...
2026-09-16 20:28:59.844490: Desired fold for training: 0
2026-09-16 20:28:59.844666: This split has 387 training an

In [1]:
import os
import glob

results_dir = "/content/nnUNet_results/Dataset001_BrainTumour/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0"

print("Checkpoint files:")
for f in glob.glob(os.path.join(results_dir, "*.pth")):
    print(os.path.basename(f), "—", round(os.path.getsize(f) / (1024**2), 1), "MB")

Checkpoint files:


In [2]:
import os
import glob

results_dir = "/content/nnUNet_results/Dataset001_BrainTumour/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0"

print("Checkpoint files:")
for f in glob.glob(os.path.join(results_dir, "*.pth")):
    print(os.path.basename(f), "—", round(os.path.getsize(f) / (1024**2), 1), "MB")

Checkpoint files:


In [3]:
!cp -r /content/nnUNet_results /content/drive/MyDrive/Medical_AI_NnUNet/

cp: cannot stat '/content/nnUNet_results': No such file or directory


In [4]:
!find /content /content/drive/MyDrive -type f \( -name "checkpoint_best.pth" -o -name "checkpoint_final.pth" \) 2>/dev/null

In [5]:
!find / -type f \( -name "checkpoint_best.pth" -o -name "checkpoint_final.pth" \) 2>/dev/null | head -20